# PART A Data Loading and Preparation

In [0]:
from pyspark.sql.functions import col

In [0]:
# Read files from Databricks Volume
df_airports = spark.read.csv(
    "/Volumes/vita/dataengineering/data/airports.csv",
    header=True,
    inferSchema=True
)

df_runways = spark.read.csv(
    "/Volumes/vita/dataengineering/data/runways.csv",
    header=True,
    inferSchema=True
)

In [0]:
# Cast columns
from pyspark.sql.functions import col

df_airports = df_airports \
    .withColumn("latitude_deg", col("latitude_deg").cast("double")) \
    .withColumn("longitude_deg", col("longitude_deg").cast("double")) \
    .withColumn("elevation_ft", col("elevation_ft").cast("int"))

df_runways = df_runways \
    .withColumn("length_ft", col("length_ft").cast("int")) \
    .withColumn("width_ft", col("width_ft").cast("int"))

display(df_airports)
display(df_runways)

In [0]:
# Print schema
print("Airports Schema:")
df_airports.printSchema()

print("Runways Schema:")
df_runways.printSchema()

In [0]:
# Count rows
airports_count = df_airports.count()
runways_count = df_runways.count()

print("Number of rows in df_airports:", airports_count)
print("Number of rows in df_runways:", runways_count)

# PART B - Join and Transforamtion

In [0]:
# Inner join airports and runways
df_combined = df_airports.join(
    df_runways,
    df_airports.ident == df_runways.airport_ident,
    "inner"
)

# Preview result
display(df_combined)

In [0]:
# Check schema
df_combined.printSchema()

In [0]:
# Row count
print("Number of rows in df_combined:", df_combined.count())

In [0]:
from pyspark.sql.functions import col, when

# Add runway_category column
df_combined = df_combined.withColumn(
    "runway_category",
    when(col("length_ft") >= 10000, "Long")
    .when((col("length_ft") >= 5000) & (col("length_ft") < 10000), "Medium")
    .otherwise("Short")
)

# Preview result
display(df_combined.select("length_ft", "runway_category"))

# Part c - Analysis Queries

In [0]:
from pyspark.sql.functions import count, col

# Top 10 countries with highest number of runways
top_10_countries = df_combined.groupBy("iso_country") \
    .agg(count("*").alias("runway_count")) \
    .orderBy(col("runway_count").desc()) \
    .limit(10)

# Display result
display(top_10_countries)

In [0]:
from pyspark.sql.functions import avg, col

# Average runway length per continent
avg_runway_length = df_combined \
    .filter(col("length_ft").isNotNull()) \
    .groupBy("continent") \
    .agg(avg("length_ft").alias("avg_length_ft")) \
    .orderBy(col("avg_length_ft").desc())

# Display result
display(avg_runway_length)

In [0]:
from pyspark.sql.functions import col, count

# Airports in India with at least one lighted runway
india_lighted_runways = df_combined \
    .filter((col("iso_country") == "IN") & (col("lighted") == "1")) \
    .groupBy("name", "municipality") \
    .agg(count("*").alias("number_of_lighted_runways")) \
    .orderBy(col("number_of_lighted_runways").desc())

# Display result
display(india_lighted_runways)

In [0]:
from pyspark.sql.functions import max, col

# Top 5 airports with longest maximum runway length
top_5_airports = df_combined \
    .filter(col("length_ft").isNotNull()) \
    .groupBy("name", "iso_country") \
    .agg(max("length_ft").alias("max_runway_length")) \
    .orderBy(col("max_runway_length").desc()) \
    .limit(5)

# Display result
display(top_5_airports)

# Part D - Save as Parquet

In [0]:
# Save df_combined as Parquet in Databricks Volume
# Drop duplicate 'id' column from runways to avoid COLUMN_ALREADY_EXISTS error
df_combined.drop(df_runways.id) \
    .write \
    .mode("overwrite") \
    .parquet("/Volumes/vita/dataengineering/data/df_combined_parquet")

In [0]:
# Read parquet back
df_check = spark.read.parquet(
    "/Volumes/vita/dataengineering/data/df_combined_parquet"
)

display(df_check)
print(df_check.count())